# Kaggle Phase 3 — Training, Baselines, Drift-Aware Adaptive Evaluation

Third of 3 notebooks. Requires TWO attached datasets (Add Input):
1. `kagglephase1-output` — features/windows/masks from `kagglephase1`
2. `kagglephase2-defs` — `model_defs.py` from `kagglephase2`

**What it does**
1. Trains the residual GRU per horizon on the burst-injected train split (static model).
2. Runs the non-learned baselines (persistence, SES with train-fit alpha).
3. Streams the injected test split chronologically twice — once frozen (static), once with
   the full drift-aware stack from the proposal: DriftMonitor (moving-average + z-score
   error checks) triggering OnlineAdapter (incremental fine-tuning on recent seen windows),
   with AdaptiveThreshold maintaining a rolling confidence band.
4. Reports two clearly separated result sets: the CLEAN (untouched) test data, and the
   burst-injected test data split into spike-affected vs. normal windows.


## Step 1: Input Checks + Imports

In [25]:
import os, sys, gc, json, time
import numpy as np
from pathlib import Path

IN_KAGGLE = os.path.exists('/kaggle')
inp = Path('/kaggle/input') if IN_KAGGLE else Path('.')

hits = sorted(inp.glob('**/features_injected.npy'), key=lambda p: len(p.parts))
if not hits:
    raise FileNotFoundError(
        "Phase-1 output not found under /kaggle/input -- attach the 'kagglephase1-output' "
        "dataset (produced by kagglephase1.ipynb) via 'Add Input', then re-run."
    )
P1 = hits[0].parent
defs_hits = sorted(inp.glob('**/model_defs.py'), key=lambda p: len(p.parts))
if not defs_hits:
    raise FileNotFoundError(
        "model_defs.py not found under /kaggle/input -- attach the 'kagglephase2-defs' "
        "dataset (produced by kagglephase2.ipynb) via 'Add Input', then re-run."
    )
sys.path.insert(0, str(defs_hits[0].parent))
from model_defs import (AdaptiveGRUModel, WindowDataset, collate_pad, make_loader, set_seed,
                        EarlyStopping, train_one_epoch, compute_metrics, predict_all, evaluate,
                        train_model, persistence_preds, ses_fit_alpha, ses_preds,
                        DriftMonitor, AdaptiveThreshold, OnlineAdapter)
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Phase-1 input : {P1}")
print(f"model_defs    : {defs_hits[0]}")
print(f"device        : {device}")

manifest = json.load(open(P1 / 'manifest.json'))
fc = json.load(open(P1 / 'feature_cols.json'))
stats = json.load(open(P1 / 'normalization_stats.json'))
TARGET_NAMES = fc['target_names']
TARGET_IDX = fc['target_idx']
TARGET_MEAN = np.array([stats[c]['mean'] for c in fc['target_columns']])
TARGET_STD  = np.array([stats[c]['std']  for c in fc['target_columns']])
FEAT_INJ, FEAT_CLEAN = P1 / 'features_injected.npy', P1 / 'features_clean.npy'
windows = {k: np.load(P1 / f'windows_{k}.npy') for k in ('train', 'val', 'test')}
windows_test_clean = np.load(P1 / 'windows_test_clean.npy')
spike_mask = np.load(P1 / 'spike_mask.npy')
feat_inj_arr = np.load(FEAT_INJ, mmap_mode='r')
feat_clean_arr = np.load(FEAT_CLEAN, mmap_mode='r')
print({k: len(v) for k, v in windows.items()}, '| clean test:', len(windows_test_clean))


Phase-1 input : /kaggle/input/datasets/thanakaran/kagglephase1-output
model_defs    : /kaggle/input/datasets/thanakaran/kagglephase2-defs/model_defs.py
device        : cuda
{'train': 142653, 'val': 33335, 'test': 33343} | clean test: 33343


## Step 2: Config

In [26]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Phase3Config:
    horizons: List[int] = field(default_factory=lambda: [1, 2, 3, 4, 5])
    hidden_size: int = 128
    num_layers: int = 2
    dropout: float = 0.2
    batch_size: int = 128       # up from 64 -- better GPU utilization on padded variable-length batches
    eval_batch_size: int = 512  # up from 256 -- no backward pass, safe to go larger
    lr: float = 1e-3
    epochs: int = 30
    patience: int = 7
    stream_chunk: int = 768     # up from 512 -- fewer, larger chunks = less per-chunk monitor overhead
    adapt_lr: float = 1e-4
    adapt_recent: int = 4096
    spike_lookback_tail: int = 60
    ckpt_dir: str = '/kaggle/working/phase3_checkpoints' if os.path.exists('/kaggle') else './phase3_checkpoints'

cfg = Phase3Config()
Path(cfg.ckpt_dir).mkdir(parents=True, exist_ok=True)
print(cfg)
print(f"NOTE: horizons scoped to {cfg.horizons}. Phase 1's windows/features and Phase 2's "
      f"model/dataset code are horizon-agnostic (horizon is always a runtime parameter) -- "
      f"changing this list requires no changes to either notebook.")
print(f"Speedups applied vs the prior run: batch_size 64->{cfg.batch_size} "
      f"(fewer, better-utilized GPU steps per epoch), eval_batch_size 256->{cfg.eval_batch_size}, "
      f"stream_chunk 512->{cfg.stream_chunk} (less per-chunk overhead in Step 5). "
      f"Training 5 horizons (1-5) instead of the original 3 [1,5,10] -- total wall-clock time "
      f"is expected to land similar to the original run, but the per-epoch speedups above still "
      f"apply and the resulting model gives a full 15-75s forecast curve instead of 3 sparse points.")


Phase3Config(horizons=[1, 2, 3, 4, 5], hidden_size=128, num_layers=2, dropout=0.2, batch_size=128, eval_batch_size=512, lr=0.001, epochs=30, patience=7, stream_chunk=768, adapt_lr=0.0001, adapt_recent=4096, spike_lookback_tail=60, ckpt_dir='/kaggle/working/phase3_checkpoints')
NOTE: horizons scoped to [1, 2, 3, 4, 5]. Phase 1's windows/features and Phase 2's model/dataset code are horizon-agnostic (horizon is always a runtime parameter) -- changing this list requires no changes to either notebook.
Speedups applied vs the prior run: batch_size 64->128 (fewer, better-utilized GPU steps per epoch), eval_batch_size 256->512, stream_chunk 512->768 (less per-chunk overhead in Step 5). Training 5 horizons (1-5) instead of the original 3 [1,5,10] -- total wall-clock time is expected to land similar to the original run, but the per-epoch speedups above still apply and the resulting model gives a full 15-75s forecast curve instead of 3 sparse points.


## Step 3: Train the Static Residual GRU (per horizon, burst-injected train split)

In [27]:
static_results = {}
for h in cfg.horizons:
    print(f"\n{'='*58}\nHORIZON {h}  ({h*15}s ahead)\n{'='*58}")
    set_seed(42 + h)
    tr_loader = make_loader(FEAT_INJ, windows['train'], h, TARGET_IDX,
                            batch_size=cfg.batch_size, shuffle=True)
    va_loader = make_loader(FEAT_INJ, windows['val'], h, TARGET_IDX,
                            batch_size=cfg.eval_batch_size)
    model = AdaptiveGRUModel(input_size=27, hidden_size=cfg.hidden_size,
                             num_layers=cfg.num_layers, dropout=cfg.dropout,
                             residual_indices=TARGET_IDX).to(device)
    print(f"  {model.n_params():,} params | train batches {len(tr_loader)} | val batches {len(va_loader)}")
    ckpt = str(Path(cfg.ckpt_dir) / f'gru_h{h}_static.pt')
    best = train_model(model, tr_loader, va_loader, device, ckpt,
                       epochs=cfg.epochs, lr=cfg.lr, patience=cfg.patience)
    model.load_state_dict(torch.load(ckpt, map_location=device)['model_state_dict'])
    model.eval()

    te_inj = make_loader(FEAT_INJ, windows['test'], h, TARGET_IDX, batch_size=cfg.eval_batch_size)
    te_cln = make_loader(FEAT_CLEAN, windows_test_clean, h, TARGET_IDX, batch_size=cfg.eval_batch_size)
    m_inj = evaluate(model, te_inj, device, TARGET_STD, TARGET_MEAN, TARGET_NAMES)
    m_cln = evaluate(model, te_cln, device, TARGET_STD, TARGET_MEAN, TARGET_NAMES)
    static_results[h] = {'best_val_loss': best, 'test_injected': m_inj, 'test_clean': m_cln}
    print(f"  clean-test MAPE mean={m_cln['mape_mean']:.2f}% | injected-test MAPE mean={m_inj['mape_mean']:.2f}%")
    del model, tr_loader, va_loader, te_inj, te_cln
    torch.cuda.empty_cache(); gc.collect()



HORIZON 1  (15s ahead)
  167,876 params | train batches 1115 | val batches 66
  epoch   1 | train=0.006615 | val=0.000512 | patience=0/7 | 112s
  epoch   5 | train=0.006134 | val=0.000345 | patience=0/7 | 556s
  epoch  10 | train=0.005833 | val=0.000482 | patience=4/7 | 1108s
  early stop at epoch 13
  done: best_val=0.000329 | 1439s
  clean-test MAPE mean=0.08% | injected-test MAPE mean=0.13%

HORIZON 2  (30s ahead)
  167,876 params | train batches 1115 | val batches 66
  epoch   1 | train=0.010480 | val=0.001050 | patience=0/7 | 111s
  epoch   5 | train=0.009259 | val=0.000803 | patience=1/7 | 557s
  epoch  10 | train=0.008183 | val=0.000671 | patience=3/7 | 1110s
  early stop at epoch 14
  done: best_val=0.000631 | 1552s
  clean-test MAPE mean=0.21% | injected-test MAPE mean=0.21%

HORIZON 3  (45s ahead)
  167,876 params | train batches 1115 | val batches 66
  epoch   1 | train=0.015125 | val=0.001447 | patience=0/7 | 112s
  epoch   5 | train=0.012654 | val=0.001359 | patience=1/7 

## Step 4: Non-Learned Baselines (persistence, SES) on Both Test Variants

In [29]:
def spike_window_mask(wins, horizon):
    tgt_rows = wins[:, 0] - 1 + horizon
    m = spike_mask[tgt_rows].copy()
    tail = cfg.spike_lookback_tail
    for i, (e, L) in enumerate(wins):
        if not m[i]:
            m[i] = spike_mask[max(e - tail, e - L):e].any()
    return m

baseline_results = {}
for h in cfg.horizons:
    row = {}
    for variant, feat_arr, wins in (('clean', feat_clean_arr, windows_test_clean),
                                    ('injected', feat_inj_arr, windows['test'])):
        tgt = np.asarray(feat_arr[wins[:, 0] - 1 + h])[:, TARGET_IDX].astype(np.float64)
        p = persistence_preds(feat_arr, wins, TARGET_IDX)
        alphas = ses_fit_alpha(feat_arr, windows['train'], h, TARGET_IDX, max_n=8000, seed=h)
        s = ses_preds(feat_arr, wins, TARGET_IDX, alphas)
        row[variant] = {
            'persistence': compute_metrics(p, tgt, TARGET_STD, TARGET_MEAN, TARGET_NAMES),
            'ses': compute_metrics(s, tgt, TARGET_STD, TARGET_MEAN, TARGET_NAMES),
            'ses_alphas': alphas,
        }
    baseline_results[h] = row
    print(f"h={h}: clean  P={row['clean']['persistence']['mape_mean']:.2f}%  "
          f"SES={row['clean']['ses']['mape_mean']:.2f}% (alpha={row['clean']['ses_alphas']}) | "
          f"injected  P={row['injected']['persistence']['mape_mean']:.2f}%  "
          f"SES={row['injected']['ses']['mape_mean']:.2f}%")


h=1: clean  P=0.05%  SES=0.06% (alpha=[np.float64(1.0), np.float64(0.9), np.float64(0.85), np.float64(0.85)]) | injected  P=0.13%  SES=0.15%
h=2: clean  P=0.11%  SES=0.12% (alpha=[np.float64(1.0), np.float64(0.85), np.float64(0.85), np.float64(0.85)]) | injected  P=0.26%  SES=0.28%
h=3: clean  P=0.16%  SES=0.17% (alpha=[np.float64(1.0), np.float64(0.75), np.float64(0.75), np.float64(0.85)]) | injected  P=0.40%  SES=0.41%
h=4: clean  P=0.21%  SES=0.21% (alpha=[np.float64(1.0), np.float64(0.6), np.float64(1.0), np.float64(1.0)]) | injected  P=0.52%  SES=0.52%
h=5: clean  P=0.24%  SES=0.24% (alpha=[np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0)]) | injected  P=0.62%  SES=0.62%


## Step 5: Streaming Drift-Aware Evaluation (injected test, chronological)

Two passes per horizon over the same chronological stream of injected-test windows:
- **static**: frozen checkpoint.
- **adaptive**: DriftMonitor watches chunk-level error (EWMA + z-score vs. a frozen
  reference window); on sustained breach, OnlineAdapter fine-tunes on recent already-seen
  windows (their targets are in the stream's past — no future information). The trailing
  10 windows are excluded from the adaptation pool to respect the max-horizon overlap.
  AdaptiveThreshold maintains a rolling P50/P90 confidence band throughout.


In [30]:
def stream_eval(h, adaptive):
    ckpt = str(Path(cfg.ckpt_dir) / f'gru_h{h}_static.pt')
    model = AdaptiveGRUModel(input_size=27, hidden_size=cfg.hidden_size,
                             num_layers=cfg.num_layers, dropout=cfg.dropout,
                             residual_indices=TARGET_IDX).to(device)
    model.load_state_dict(torch.load(ckpt, map_location=device)['model_state_dict'])
    model.eval()

    wins = windows['test'][np.argsort(windows['test'][:, 0], kind='stable')]
    monitor = DriftMonitor()
    athresh = AdaptiveThreshold()
    adapter = OnlineAdapter(FEAT_INJ, TARGET_IDX, h, lr=cfg.adapt_lr,
                            recent=cfg.adapt_recent, batch_size=cfg.batch_size)
    preds, tgts = [], []
    triggers, bands = [], []
    n_chunks = int(np.ceil(len(wins) / cfg.stream_chunk))
    for ci in range(n_chunks):
        chunk = wins[ci * cfg.stream_chunk:(ci + 1) * cfg.stream_chunk]
        loader = make_loader(FEAT_INJ, chunk, h, TARGET_IDX, batch_size=cfg.eval_batch_size)
        p, t = predict_all(model, loader, device)
        preds.append(p); tgts.append(t)
        err = float(((p - t) ** 2).mean())
        lo, hi = athresh.update(np.abs(p - t).mean(axis=1))
        bands.append((lo, hi))
        if adaptive and monitor.update(err):
            seen = wins[:ci * cfg.stream_chunk + len(chunk)][:-10]
            if len(seen) >= 512:
                adapter.adapt(model, seen, device)
                triggers.append(ci)
                monitor.hits = 0
    preds, tgts = np.concatenate(preds), np.concatenate(tgts)
    m_all = compute_metrics(preds, tgts, TARGET_STD, TARGET_MEAN, TARGET_NAMES)
    sm = spike_window_mask(wins, h)
    m_spike  = compute_metrics(preds[sm], tgts[sm], TARGET_STD, TARGET_MEAN, TARGET_NAMES)
    m_normal = compute_metrics(preds[~sm], tgts[~sm], TARGET_STD, TARGET_MEAN, TARGET_NAMES)
    del model
    torch.cuda.empty_cache(); gc.collect()
    return {'overall': m_all, 'spike': m_spike, 'normal': m_normal,
            'n_spike_windows': int(sm.sum()), 'n_windows': len(wins),
            'adapt_triggers': triggers, 'n_updates': adapter.n_updates,
            'final_band': bands[-1]}

stream_results = {}
for h in cfg.horizons:
    print(f"\n--- horizon {h}: static pass ---")
    t0 = time.time()
    r_static = stream_eval(h, adaptive=False)
    print(f"  overall MAPE={r_static['overall']['mape_mean']:.2f}% | "
          f"spike={r_static['spike']['mape_mean']:.2f}% ({r_static['n_spike_windows']} windows) | "
          f"normal={r_static['normal']['mape_mean']:.2f}% | {time.time()-t0:.0f}s")
    print(f"--- horizon {h}: adaptive pass ---")
    t0 = time.time()
    r_adapt = stream_eval(h, adaptive=True)
    print(f"  overall MAPE={r_adapt['overall']['mape_mean']:.2f}% | "
          f"spike={r_adapt['spike']['mape_mean']:.2f}% | normal={r_adapt['normal']['mape_mean']:.2f}% | "
          f"updates={r_adapt['n_updates']} at chunks {r_adapt['adapt_triggers']} | {time.time()-t0:.0f}s")
    stream_results[h] = {'static': r_static, 'adaptive': r_adapt}



--- horizon 1: static pass ---
  overall MAPE=0.13% | spike=0.24% (10028 windows) | normal=0.09% | 9s
--- horizon 1: adaptive pass ---
  overall MAPE=0.12% | spike=0.23% | normal=0.07% | updates=3 at chunks [11, 13, 15] | 17s

--- horizon 2: static pass ---
  overall MAPE=0.21% | spike=0.35% (10119 windows) | normal=0.15% | 8s
--- horizon 2: adaptive pass ---
  overall MAPE=0.20% | spike=0.35% | normal=0.14% | updates=8 at chunks [10, 12, 14, 16, 21, 23, 25, 32] | 29s

--- horizon 3: static pass ---
  overall MAPE=0.26% | spike=0.46% (10210 windows) | normal=0.17% | 8s
--- horizon 3: adaptive pass ---
  overall MAPE=0.26% | spike=0.44% | normal=0.18% | updates=9 at chunks [10, 12, 14, 16, 20, 22, 24, 26, 28] | 32s

--- horizon 4: static pass ---
  overall MAPE=0.35% | spike=0.62% (10300 windows) | normal=0.23% | 8s
--- horizon 4: adaptive pass ---
  overall MAPE=0.34% | spike=0.57% | normal=0.24% | updates=9 at chunks [10, 12, 14, 16, 20, 22, 24, 26, 28] | 32s

--- horizon 5: static p

## Step 6: Final Comparison Tables + Save

In [31]:
import zipfile

print("="*74)
print("TABLE A -- CLEAN (untouched) test data, MAPE % per target")
print("="*74)
for h in cfg.horizons:
    print(f"\nhorizon {h} ({h*15}s ahead)")
    print(f"  {'target':<18} {'Persist':>9} {'SES':>9} {'GRU-static':>11}")
    b = baseline_results[h]['clean']
    g = static_results[h]['test_clean']
    for n in TARGET_NAMES:
        print(f"  {n:<18} {b['persistence'][n]['mape']:>8.2f}% {b['ses'][n]['mape']:>8.2f}% "
              f"{g[n]['mape']:>10.2f}%")

print()
print("="*74)
print("TABLE B -- BURST-INJECTED test data (streaming), MAPE % per target")
print("="*74)
for h in cfg.horizons:
    b = baseline_results[h]['injected']
    sr = stream_results[h]
    for regime in ('spike', 'normal'):
        print(f"\nhorizon {h}, {regime.upper()} windows "
              f"({sr['static']['n_spike_windows'] if regime=='spike' else sr['static']['n_windows']-sr['static']['n_spike_windows']}):")
        print(f"  {'target':<18} {'Persist*':>9} {'SES*':>9} {'GRU-static':>11} {'GRU-adaptive':>13}")
        for n in TARGET_NAMES:
            print(f"  {n:<18} {b['persistence'][n]['mape']:>8.2f}% {b['ses'][n]['mape']:>8.2f}% "
                  f"{sr['static'][regime][n]['mape']:>10.2f}% {sr['adaptive'][regime][n]['mape']:>12.2f}%")
print("\n(* baseline columns are whole-injected-test figures; the GRU columns are per-regime.)")

print()
print("="*74)
print("VERDICT SUMMARY")
print("="*74)
for h in cfg.horizons:
    sr = stream_results[h]
    d_spike = sr['static']['spike']['mape_mean'] - sr['adaptive']['spike']['mape_mean']
    d_norm  = sr['static']['normal']['mape_mean'] - sr['adaptive']['normal']['mape_mean']
    print(f"h={h}: adaptation changed spike-window MAPE by {-d_spike:+.2f} pts "
          f"(static {sr['static']['spike']['mape_mean']:.2f}% -> adaptive {sr['adaptive']['spike']['mape_mean']:.2f}%), "
          f"normal windows by {-d_norm:+.2f} pts, with {sr['adaptive']['n_updates']} online update(s)")
print()
print("Positive improvements on spike windows with non-degraded normal windows = the")
print("drift-aware mechanism is doing its job. Report whatever the numbers actually show.")

out_dir = Path('/kaggle/working') if os.path.exists('/kaggle') else Path('.')
results = {'config': {k: getattr(cfg, k) for k in ('horizons', 'hidden_size', 'num_layers',
                                                     'batch_size', 'lr', 'epochs', 'stream_chunk',
                                                     'adapt_lr', 'adapt_recent')},
           'static': static_results, 'baselines': baseline_results, 'streaming': stream_results}
with open(out_dir / 'phase3_results.json', 'w') as f:
    json.dump(results, f, indent=1, default=float)
zip_path = out_dir / 'kagglephase3_output.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(out_dir / 'phase3_results.json', 'phase3_results.json')
    for p in Path(cfg.ckpt_dir).glob('*.pt'):
        z.write(p, f'checkpoints/{p.name}')
print(f"Saved {out_dir / 'phase3_results.json'} and {zip_path} -- download from the Output tab.")


TABLE A -- CLEAN (untouched) test data, MAPE % per target

horizon 1 (15s ahead)
  target               Persist       SES  GRU-static
  cpu_usage              0.01%     0.01%       0.04%
  mem_usage              0.07%     0.07%       0.09%
  mem_working_set        0.07%     0.08%       0.09%
  mem_rss                0.07%     0.09%       0.10%

horizon 2 (30s ahead)
  target               Persist       SES  GRU-static
  cpu_usage              0.03%     0.03%       0.21%
  mem_usage              0.13%     0.14%       0.20%
  mem_working_set        0.13%     0.14%       0.20%
  mem_rss                0.15%     0.16%       0.22%

horizon 3 (45s ahead)
  target               Persist       SES  GRU-static
  cpu_usage              0.04%     0.04%       0.15%
  mem_usage              0.20%     0.21%       0.22%
  mem_working_set        0.20%     0.21%       0.22%
  mem_rss                0.22%     0.23%       0.24%

horizon 4 (60s ahead)
  target               Persist       SES  GRU-static
  